# DESI DR1 Lyα delta files — visual sanity check

Each `delta-XXXX.fits` (where `XXXX` is the HEALPix pixel ID) contains:
- `LAMBDA`  — common observed-frame wavelength grid (1D, Å)
- `METADATA` — per-LOS table: `LOS_ID, RA, DEC, Z, MEANSNR, TARGETID, NIGHT, PETAL, TILE`
- `DELTA_BLIND` — (n_LOS, n_λ) array of δ_F = F/⟨F⟩ − 1 in the Lyα forest
- `WEIGHT` — pipeline weights (same shape)
- `CONT` — fitted continuum × mean transmission (same shape)

Goal: load each file, inspect metadata, and plot a handful of LOS deltas to confirm the data looks sensible.

In [ ]:
import glob
import os

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

LYA = 1215.67  # Å, Lyman-α rest wavelength
WAVE_RANGE = (3600.0, 4000.0)  # Å, observed-frame window to inspect

files = sorted(glob.glob('data/delta-*.fits'))
files

In [ ]:
def load_delta(path):
    """Return (lam, meta, delta, weight, cont) for a DESI delta file."""
    with fits.open(path) as f:
        lam = f['LAMBDA'].data.astype(float)
        meta = f['METADATA'].data
        delta = f['DELTA_BLIND'].data.astype(float)
        weight = f['WEIGHT'].data.astype(float)
        cont = f['CONT'].data.astype(float)
    return lam, meta, delta, weight, cont

# quick summary table
for fn in files:
    lam, meta, delta, weight, cont = load_delta(fn)
    valid = weight > 0
    print(f"{os.path.basename(fn):20s}  n_LOS={len(meta):4d}  "
          f"z=[{meta['Z'].min():.2f},{meta['Z'].max():.2f}]  "
          f"λ=[{lam.min():.0f},{lam.max():.0f}]Å  "
          f"⟨δ⟩={delta[valid].mean():+.3f}  σ(δ)={delta[valid].std():.3f}")

## Example delta spectra

For each file we pick the 5 LOS with the highest mean SNR and plot δ vs. observed wavelength. A healthy Lyα forest delta should fluctuate around 0 with O(1) excursions (more absorption → more negative δ); pixels with weight = 0 are masked.

In [ ]:
N_SHOW = 5

fig, axes = plt.subplots(len(files), 1, figsize=(10, 2.2 * len(files)),
                         sharex=True, constrained_layout=True)
if len(files) == 1:
    axes = [axes]

in_window = None  # set per-file below

for ax, fn in zip(axes, files):
    lam, meta, delta, weight, _ = load_delta(fn)
    in_window = (lam >= WAVE_RANGE[0]) & (lam <= WAVE_RANGE[1])
    # rank by SNR but only count LOS that actually cover the window
    coverage = (weight[:, in_window] > 0).sum(axis=1)
    candidates = np.where(coverage > 0)[0]
    order = candidates[np.argsort(meta['MEANSNR'][candidates])[::-1]][:N_SHOW]
    for i in order:
        d = np.where(weight[i] > 0, delta[i], np.nan)
        ax.plot(lam[in_window], d[in_window], lw=0.6,
                label=f"LOS {meta['LOS_ID'][i]}  z={meta['Z'][i]:.2f}  SNR={meta['MEANSNR'][i]:.1f}")
    ax.axhline(0, color='k', lw=0.5, alpha=0.5)
    ax.set_xlim(*WAVE_RANGE)
    ax.set_ylim(-1.5, 1.5)
    ax.set_ylabel(r'$\delta_F$')
    ax.set_title(os.path.basename(fn), loc='left', fontsize=9)
    ax.legend(fontsize=7, ncol=1, loc='upper right')

axes[-1].set_xlabel(r'observed wavelength $\lambda$ [Å]')
plt.show()

## Continuum × ⟨F⟩ for the same LOS

The `CONT` extension stores the fitted quasar continuum times the mean transmission, i.e. the quantity that was divided out to obtain δ. Plotting it confirms the continuum is smooth and the absorption lives in δ.

In [ ]:
fig, axes = plt.subplots(len(files), 1, figsize=(10, 2.2 * len(files)),
                         sharex=True, constrained_layout=True)
if len(files) == 1:
    axes = [axes]

for ax, fn in zip(axes, files):
    lam, meta, _, weight, cont = load_delta(fn)
    in_window = (lam >= WAVE_RANGE[0]) & (lam <= WAVE_RANGE[1])
    coverage = (weight[:, in_window] > 0).sum(axis=1)
    candidates = np.where(coverage > 0)[0]
    order = candidates[np.argsort(meta['MEANSNR'][candidates])[::-1]][:N_SHOW]
    for i in order:
        c = np.where(weight[i] > 0, cont[i], np.nan)
        ax.plot(lam[in_window], c[in_window], lw=0.6,
                label=f"LOS {meta['LOS_ID'][i]}  z={meta['Z'][i]:.2f}")
    ax.set_xlim(*WAVE_RANGE)
    ax.set_ylabel(r'$C \cdot \langle F \rangle$')
    ax.set_title(os.path.basename(fn), loc='left', fontsize=9)
    ax.legend(fontsize=7, ncol=1, loc='upper right')

axes[-1].set_xlabel(r'observed wavelength $\lambda$ [Å]')
plt.show()

## Distributions: redshift, SNR, and sky positions

Combined view across all loaded files — useful to confirm the HEALPix pixels sit where expected and that the QSO redshift / SNR distributions look reasonable.

In [ ]:
all_meta = {fn: load_delta(fn)[1] for fn in files}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), constrained_layout=True)

for fn, m in all_meta.items():
    label = os.path.basename(fn).replace('delta-', '').replace('.fits', '')
    axes[0].hist(m['Z'], bins=np.linspace(2, 4.5, 26), histtype='step', label=label)
    axes[1].hist(np.log10(np.clip(m['MEANSNR'], 1e-2, None)),
                 bins=30, histtype='step', label=label)
    axes[2].scatter(m['RA'], m['DEC'], s=4, alpha=0.6, label=label)

axes[0].set_xlabel('QSO redshift')
axes[0].set_ylabel('count')
axes[0].legend(fontsize=8, title='HEALPix')

axes[1].set_xlabel(r'$\log_{10}$ MEANSNR')
axes[1].set_ylabel('count')

axes[2].set_xlabel('RA [deg]')
axes[2].set_ylabel('Dec [deg]')
axes[2].set_aspect('equal', adjustable='datalim')

plt.show()

## Stacked δ along the forest

Averaging δ over all LOS in a single file should give something close to zero (by construction). Large systematic departures from 0 would be a red flag.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for fn in files:
    lam, meta, delta, weight, _ = load_delta(fn)
    in_window = (lam >= WAVE_RANGE[0]) & (lam <= WAVE_RANGE[1])
    w = np.where(weight > 0, weight, 0.0)
    wsum = w.sum(axis=0)
    mean_delta = np.where(wsum > 0,
                          (w * delta).sum(axis=0) / np.where(wsum > 0, wsum, 1),
                          np.nan)
    label = os.path.basename(fn).replace('delta-', '').replace('.fits', '')
    ax.plot(lam[in_window], mean_delta[in_window], lw=0.7,
            label=f'pix {label}  ({len(meta)} LOS)')

ax.axhline(0, color='k', lw=0.5)
ax.set_xlim(*WAVE_RANGE)
ax.set_xlabel(r'observed wavelength $\lambda$ [Å]')
ax.set_ylabel(r'weighted $\langle \delta_F \rangle$')
ax.set_ylim(-0.3, 0.3)
ax.legend(fontsize=8)
plt.show()

## SNR distribution of training-usable spectra

These deltas are destined for neural-net training, so every pixel in the 3600–4000 Å window must carry a value. No LOS is naturally NaN-free there (sky-line masking leaves interior gaps), so "usable" is defined as:

1. the forest **spans the full window** — finite `DELTA_BLIND` at both the 3600 Å and 4000 Å edge pixels (otherwise we'd be fabricating a large edge block), **and**
2. the **interior NaN fraction is below `MAX_INTERP_FRAC`** — short masked gaps that can be honestly linearly interpolated.

Qualifying spectra then have their interior NaNs interpolated so the window is fully valued, giving a clean `(n_usable, n_pix)` array `usable_spectra` ready for training. The histogram shows the `MEANSNR` distribution of those spectra. Dropped spectra are counted and broken down by reason.

In [ ]:
MAX_INTERP_FRAC = 0.20  # drop a LOS if more than this fraction of the window must be interpolated


def fill_interior_nans(row, x):
    """Linearly interpolate NaNs in `row` (assumes both ends are finite)."""
    good = np.isfinite(row)
    out = row.copy()
    out[~good] = np.interp(x[~good], x[good], row[good])
    return out


snr_usable = []
usable_spectra = []      # filled, NaN-free deltas in the window (the NN inputs)
n_total = 0
n_no_coverage = 0        # forest does not reach across the window
n_too_masked = 0         # spans window but too much to interpolate

for fn in files:
    lam, meta, delta, _, _ = load_delta(fn)
    in_window = (lam >= WAVE_RANGE[0]) & (lam <= WAVE_RANGE[1])
    lam_w = lam[in_window]
    dw = delta[:, in_window]
    fin = np.isfinite(dw)

    spans = fin[:, 0] & fin[:, -1]                 # finite at both window edges
    interp_frac = (~fin).sum(axis=1) / fin.shape[1]
    usable = spans & (interp_frac <= MAX_INTERP_FRAC)

    n_total += len(meta)
    n_no_coverage += int((~spans).sum())
    n_too_masked += int((spans & (interp_frac > MAX_INTERP_FRAC)).sum())

    for i in np.where(usable)[0]:
        usable_spectra.append(fill_interior_nans(dw[i], lam_w))
    snr_usable.append(np.asarray(meta['MEANSNR'])[usable])

snr_usable = np.concatenate(snr_usable)
usable_spectra = np.array(usable_spectra)

n_usable = len(snr_usable)
print(f"Total spectra (LOS):                 {n_total}")
print(f"Usable (full window, interpolated):  {n_usable}")
print(f"  -> resulting array shape:          {usable_spectra.shape}, "
      f"contains NaNs: {np.isnan(usable_spectra).any()}")
print(f"Dropped:                             {n_total - n_usable}")
print(f"  - forest does not span window:     {n_no_coverage}")
print(f"  - >{MAX_INTERP_FRAC:.0%} of window masked:        {n_too_masked}")

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
ax.hist(snr_usable, bins=40, color='steelblue', edgecolor='k', linewidth=0.4)
ax.set_xlabel('MEANSNR')
ax.set_ylabel('number of spectra')
ax.set_title(f'SNR distribution of training-usable spectra '
             f'({n_usable} of {n_total}, {WAVE_RANGE[0]:.0f}–{WAVE_RANGE[1]:.0f} Å)')
plt.show()

## Scan: usable spectra vs. wavelength window

Only 180 LOS span 3600–4000 Å because that blue window sits below most forests. Here we scan the window to see how the training-usable count responds, reusing the exact same criterion (`fill_interior_nans` + `MAX_INTERP_FRAC`). Two sweeps:

- **left:** a fixed-width window slid in observed wavelength (usable count vs. window centre, one curve per width);
- **right:** the window anchored at the blue end (3600 Å) and grown redward (usable count vs. width).

Use it to pick a window that balances spectral coverage against training-set size.

In [ ]:
# pre-load once so the scan does not re-read the FITS files for every window
_cache = [(lam, np.asarray(meta['MEANSNR']), delta)
          for lam, meta, delta, _, _ in (load_delta(fn) for fn in files)]


def count_usable(lo, hi, max_interp_frac=MAX_INTERP_FRAC):
    """Number of LOS usable in [lo, hi] under the training criterion."""
    n = 0
    for lam, _snr, delta in _cache:
        win = (lam >= lo) & (lam <= hi)
        if win.sum() < 2:
            continue
        fin = np.isfinite(delta[:, win])
        spans = fin[:, 0] & fin[:, -1]
        interp_frac = (~fin).sum(axis=1) / fin.shape[1]
        n += int(np.count_nonzero(spans & (interp_frac <= max_interp_frac)))
    return n


lam_min = min(c[0].min() for c in _cache)
lam_max = max(c[0].max() for c in _cache)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

# --- left: slide a fixed-width window across observed wavelength ---
widths = [200, 400, 600, 800]
for W in widths:
    starts = np.arange(lam_min, lam_max - W, 50.0)
    centres = starts + W / 2
    counts = [count_usable(s, s + W) for s in starts]
    axL.plot(centres, counts, lw=1.3, label=f'{W:.0f} Å wide')
axL.axvline(np.mean(WAVE_RANGE), color='k', ls='--', lw=0.8,
            label=f'current ({WAVE_RANGE[0]:.0f}–{WAVE_RANGE[1]:.0f} Å)')
axL.set_xlabel('window centre [Å]')
axL.set_ylabel('usable spectra')
axL.set_title('fixed-width window slid in wavelength')
axL.legend(fontsize=8)

# --- right: anchor at the blue end and grow the window redward ---
widths_grow = np.arange(100, int(lam_max - WAVE_RANGE[0]), 50.0)
counts_grow = [count_usable(WAVE_RANGE[0], WAVE_RANGE[0] + w) for w in widths_grow]
axR.plot(widths_grow, counts_grow, lw=1.5, color='crimson')
axR.axvline(WAVE_RANGE[1] - WAVE_RANGE[0], color='k', ls='--', lw=0.8,
            label=f'current width ({WAVE_RANGE[1] - WAVE_RANGE[0]:.0f} Å)')
axR.set_xlabel(f'window width [Å]  (anchored at {WAVE_RANGE[0]:.0f} Å)')
axR.set_ylabel('usable spectra')
axR.set_title('window grown redward from the blue end')
axR.legend(fontsize=8)

plt.show()

# quick text summary of a few candidate windows
print('candidate window            usable')
for lo, hi in [(3600, 4000), (3700, 4100), (3800, 4200), (4000, 4400),
               (3600, 4200), (3600, 4400)]:
    print(f'  {lo:.0f}-{hi:.0f} Å ({hi - lo:.0f} Å)     {count_usable(lo, hi):4d}')

## Coverage map: where each spectrum has valid pixels

Every LOS (all files share the same 3600–5772 Å grid) becomes one horizontal row: **white** where the pixel is usable (non-NaN) and **black** where it is masked / missing. Rows are sorted by the wavelength of their first usable pixel, with ties broken by the last usable pixel, so the coverage band reads as a clean diagonal showing how the forest marches redward with QSO redshift. The dashed lines mark the 3600–4000 Å training window.

In [ ]:
# stack every LOS into one (n_LOS, n_pix) array on the shared wavelength grid
lam_grid = load_delta(files[0])[0]
all_delta = np.concatenate([load_delta(fn)[2] for fn in files], axis=0)
finite = np.isfinite(all_delta)              # True = usable pixel
n_pix = finite.shape[1]

# first / last usable pixel index per spectrum (n_pix sentinel = no data at all)
first_valid = np.where(finite.any(axis=1), finite.argmax(axis=1), n_pix)
last_valid = np.where(finite.any(axis=1),
                      n_pix - 1 - finite[:, ::-1].argmax(axis=1), -1)

# sort by first usable pixel, breaking ties by last usable pixel
order = np.lexsort((last_valid, first_valid))

# binary image: white where usable, black where not
img = finite[order].astype(float)

fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)
ax.imshow(img, aspect='auto', origin='lower', cmap='gray', vmin=0, vmax=1,
          extent=[lam_grid.min(), lam_grid.max(), 0, img.shape[0]],
          interpolation='nearest')
for edge in WAVE_RANGE:
    ax.axvline(edge, color='lime', ls='--', lw=1.0)
ax.set_xlabel(r'observed wavelength $\lambda$ [Å]')
ax.set_ylabel('spectrum (sorted by first, then last usable pixel)')
ax.set_title(f'Per-pixel coverage of {img.shape[0]} spectra '
             '(white = usable, black = masked / no data)')

plt.savefig("plots/DESI_spectra_coverage.pdf", format="pdf")
#plt.show()

## Same coverage map for the SDSS BOSS DR9 spectra

The Transformer pipeline (`dataset_functions.get_sdss_spectra`) loads SDSS Lyα spectra from `speclya-PLATE-MJD-FIBER.fits` and treats a pixel as usable via `good_pixel = (MASK_COMB == 0) & (IVAR > 0)`. Unlike the DESI delta files, these store the **full optical spectrum** (≈3566–10325 Å on a shared log-λ grid, Δlog₁₀λ = 1e-4), and the pipeline *discards* any spectrum with a NaN in its forest window — so to see coverage we instead keep the gaps and stack the raw `good_pixel` masks.

Same rendering as above: one row per spectrum, **white = usable**, **black = masked / no data**, sorted by first then last usable pixel. Dashed lines mark the pipeline's 3600–3950 Å SDSS forest extraction window. Because BOSS spectra cover the whole optical, expect near-full (white) coverage with a blue start-edge staircase and common masked sky-line columns, rather than the redshift-driven diagonal seen for DESI.

In [ ]:
from astropy.table import Table
import tqdm

# --- config (mirrors dataset_functions.get_sdss_spectra) ---
SDSS_CAT = 'data/BOSSLyaDR9_cat.fits'                              # PLATE/MJD/FIBERID/SNR catalogue
SDSS_BASE = '/virgotng/mpia/obs/SDSS/BOSSLyaDR9_spectra'          # speclya basepath
SDSS_WAVE_RANGE = (lam_grid.min(), lam_grid.max())  # blue display window covering the Lyα forest region
SDSS_FOREST_WINDOW = (3600.0, 4000.0)  # pipeline's extraction window (min/max_wavelength)
N_SDSS = 1234                       # number of catalogue spectra to load (raise for more; ~minutes)
DLOG = 1e-4                         # SDSS log10(λ) pixel spacing

# pmf list straight from the catalogue (mirrors get_sdss_file_ids, no SNR cut)
cat = Table.read(SDSS_CAT)
pmf_list = [(r['PLATE'], r['MJD'], r['FIBERID']) for r in cat[:N_SDSS]]

# common log-λ grid over the display window
glog = np.arange(np.log10(SDSS_WAVE_RANGE[0]), np.log10(SDSS_WAVE_RANGE[1]) + DLOG / 2, DLOG)
sdss_lam = 10 ** glog
n_grid = len(glog)

sdss_cov = np.zeros((len(pmf_list), n_grid), dtype=bool)   # True = usable pixel
n_loaded = n_missing = 0
for plate, mjd, fiber in tqdm.tqdm(pmf_list):
    fp = f"{SDSS_BASE}/{plate}/speclya-{plate}-{mjd}-{fiber:04d}.fits"
    try:
        with fits.open(fp) as hdul:
            d = hdul[1].data
            loglam = d['LOGLAM']
            good_pixel = (d['MASK_COMB'] == 0) & (d['IVAR'] > 0)   # same rule as the pipeline
    except (FileNotFoundError, OSError):
        n_missing += 1
        continue
    idx = np.round((loglam - glog[0]) / DLOG).astype(int)         # map onto the shared grid
    sel = (idx >= 0) & (idx < n_grid) & good_pixel
    sdss_cov[n_loaded, idx[sel]] = True
    n_loaded += 1
sdss_cov = sdss_cov[:n_loaded]
print(f"SDSS spectra loaded: {n_loaded}   (missing files skipped: {n_missing})")

# sort by first usable pixel, ties broken by last usable pixel (same as the DESI map)
fv = np.where(sdss_cov.any(axis=1), sdss_cov.argmax(axis=1), n_grid)
lv = np.where(sdss_cov.any(axis=1), n_grid - 1 - sdss_cov[:, ::-1].argmax(axis=1), -1)
order = np.lexsort((lv, fv))
img = sdss_cov[order].astype(float)

fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)
ax.imshow(img, aspect='auto', origin='lower', cmap='gray', vmin=0, vmax=1,
          extent=[sdss_lam.min(), sdss_lam.max(), 0, img.shape[0]],
          interpolation='nearest')
for edge in SDSS_FOREST_WINDOW:
    ax.axvline(edge, color='lime', ls='--', lw=1.0)
ax.set_xlabel(r'observed wavelength $\lambda$ [Å]')
ax.set_ylabel('spectrum (sorted by first, then last usable pixel)')
ax.set_title(f'SDSS BOSS DR9: per-pixel coverage of {img.shape[0]} spectra '
             '(white = usable, black = masked / no data)')

plt.savefig("plots/SDSS_spectra_coverage.pdf", format="pdf")
#plt.show()